# 21 — Sync vLLM Agentic GRPO learning runbook

Полный object-first notebook для запуска Agentic GRPO с Tunix `GRPOLearner`, sync vLLM rollout engine, обновлением actor weights, локальным JSONL logging, validation evidence и checkpoint artifact references.

Notebook ничего не скачивает неявно: Qwen snapshot берётся из `configs/grpo/qwen_agentic_local.yaml`, sync vLLM knobs — из `configs/generation/qwen_vllm_sync.yaml`. Heavy cells запускают реальные веса и training update; если preflight падает, исправляем окружение/конфиг до запуска learner.

## Placement note: vLLM GPU, env/device policy, host batching

Для отдельных batched rollout notebooks (`17`, `20`) мы явно держим JAX env на CPU через `EnvironmentDevicePolicy(backend='cpu')` и ускоряем host-side preparation через `HostBatchPolicy(prompt_workers=...)`: это снижает лишние копии и не забивает GPU, который нужен vLLM. Этот Agentic GRPO notebook использует Tunix `RLCluster`/`GRPOLearner`, поэтому не выставляет `JAX_PLATFORMS=cpu` внутри kernel: actor/reference и topology preflight должны видеть JAX accelerator. Если нужна CPU-only проверка env/device policy, используйте notebook `20_env_device_policy_benchmark.ipynb` или запускайте отдельный kernel/process с `JAX_PLATFORMS=cpu` до импорта JAX.


In [ ]:
import os

# Must be the first code cell. Restart the kernel after changing these.
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
# Do not force JAX_PLATFORMS=cpu here: actor/reference weights are JAX/Tunix objects.
# If you deliberately run CPU-only preflight, set JAX_PLATFORMS before launching Jupyter.

print('XLA_PYTHON_CLIENT_PREALLOCATE=', os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'])
print('VLLM_WORKER_MULTIPROC_METHOD=', os.environ['VLLM_WORKER_MULTIPROC_METHOD'])
print('JAX_PLATFORMS=', os.environ.get('JAX_PLATFORMS'))


In [ ]:
from pathlib import Path

import jax

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

PROFILE_PATH = ROOT / 'configs/grpo/qwen_agentic_local.yaml'
print('repo:', ROOT)
print('profile:', PROFILE_PATH)
print('jax backend:', jax.default_backend())
print('jax devices:', jax.devices())


In [ ]:

import json

from tunix_craftext.artifacts.observability import (
    JsonlRunLogger,
    MetricRecord,
    RunArtifact,
    ValidationTrajectoryRecord,
    checkpoint_artifact,
    read_jsonl,
    validation_trajectory_artifact,
)
from tunix_craftext.training.grpo_profile import (
    build_grpo_evidence_manifest,
    load_agentic_grpo_profile,
    resolve_profile_path,
)

profile = load_agentic_grpo_profile(PROFILE_PATH)
environment_config_path = resolve_profile_path(PROFILE_PATH, profile.environment_config, repo_root=ROOT)
topology_config_path = resolve_profile_path(PROFILE_PATH, profile.topology_config, repo_root=ROOT)
generation_config_path = resolve_profile_path(PROFILE_PATH, profile.generation_config, repo_root=ROOT)
snapshot_path = resolve_profile_path(PROFILE_PATH, profile.model.snapshot, repo_root=ROOT)
run_dir = resolve_profile_path(PROFILE_PATH, profile.evidence.root, repo_root=ROOT)
checkpoint_root_path = resolve_profile_path(PROFILE_PATH, profile.evidence.checkpoints, repo_root=ROOT)
provenance_path = resolve_profile_path(PROFILE_PATH, profile.evidence.provenance, repo_root=ROOT)

run_dir.mkdir(parents=True, exist_ok=True)
logger = JsonlRunLogger(run_dir)
manifest = build_grpo_evidence_manifest(profile, profile_path=PROFILE_PATH, repo_root=ROOT)
provenance_path.parent.mkdir(parents=True, exist_ok=True)
provenance_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

logger.log_artifact(RunArtifact(profile.run.name, str(provenance_path), 'profile', step=0))
logger.log_metric(
    MetricRecord(
        run_id=profile.run.name,
        step=0,
        split='benchmark',
        phase='profile_loaded',
        metrics={
            'max_steps': profile.workload.max_steps,
            'num_generations': profile.workload.num_generations,
            'max_concurrency': profile.workload.max_concurrency,
        },
    )
)
print('environment config:', environment_config_path)
print('generation config:', generation_config_path)
print('snapshot:', snapshot_path)
print('run_dir:', run_dir)
print('provenance:', provenance_path)
print('metrics:', logger.metrics_path)


In [ ]:

from dataclasses import replace

from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.inference import VllmInferenceEngine, load_generation_pipeline_config

config = load_mvp_config(environment_config_path)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=profile.run.goal,
)
validation_tasks = task_sampler.validation_tasks(seed=profile.run.seed, limit=2)
generation = load_generation_pipeline_config(generation_config_path)
if not snapshot_path.is_dir():
    raise FileNotFoundError(f'Local model snapshot is missing: {snapshot_path}')

# Tunix AgenticRLLearner requires vLLM server mode for rollout_engine='vllm'.
sync_vllm_generation = replace(
    generation.tunix,
    engine='vllm',
    vllm_server_mode=True,
    vllm_async_scheduling=False,
    vllm_init_with_random_weights=False,
    vllm_model_version=generation.tunix.vllm_model_version or 'qwen2.5-0.5b',
)
print('snapshot:', snapshot_path)
print('generation config:', generation_config_path)
print('sync vLLM Tunix rollout kwargs:', sync_vllm_generation.to_tunix_rollout_kwargs())
print('validation tasks:', validation_tasks)


## Sync vLLM smoke before loading Tunix actor/reference

Эта ячейка проверяет локальный snapshot и обычный sync vLLM backend contract. Training ниже использует Tunix `RLCluster` с vLLM server-mode rollout, но smoke дешевле и даёт раннюю ошибку, если snapshot/vLLM setup сломан.


In [ ]:

from dataclasses import replace

from tunix_craftext.env.prompts import RenderedPrompt
from tunix_craftext.inference import GenerationBatch, VllmInferenceEngine, collect_generation_results_sync
from tunix_craftext.models.llm import LlmRequest

engine_profile = replace(
    generation.profile,
    model=str(snapshot_path),
    metadata={**dict(generation.profile.metadata), 'notebook': '21_sync_vllm_grpo_learning.ipynb'},
)
vllm_engine = VllmInferenceEngine.from_profile(engine_profile)
smoke_prompt = RenderedPrompt(
    """<|im_start|>user
Return exactly <action>NOOP</action><|im_end|>
<|im_start|>assistant
""",
    runtime.actions,
    'sync-vllm-smoke',
)
smoke_records = collect_generation_results_sync(
    vllm_engine,
    (
        GenerationBatch(
            (LlmRequest(smoke_prompt, max_new_tokens=4),),
            group_id='sync-vllm-smoke',
            policy_version=0,
        ),
    ),
)
smoke_text = smoke_records[0].result.responses[0].raw_text
logger.log_metric(
    MetricRecord(
        run_id=profile.run.name,
        step=0,
        split='benchmark',
        phase='sync_vllm_smoke',
        metrics={'responses': len(smoke_records[0].result.responses)},
    )
)
print('sync vLLM smoke:', smoke_text)


## Validation evidence before train

Validation здесь — не fake metric: это короткая deterministic CrafText tool-loop trajectory, записанная как artifact. Она проверяет environment/tool/reward path до дорогого GRPO update.


In [ ]:

from tunix_craftext.artifacts.observability import RunArtifact
from tunix_craftext.training.agentic_grpo_smoke import collect_scripted_grpo_group, save_scripted_grpo_smoke

pre_validation_path = run_dir / 'validation' / 'pre-train-scripted-grpo.json'
scripted_results = await collect_scripted_grpo_group(
    config_path=environment_config_path,
    goal=profile.run.goal,
    seed=profile.run.seed,
    group_id=0,
    action_sequences=(('NOOP',) * 2, ('LEFT',) * 2),
    horizon=2,
)
save_scripted_grpo_smoke(pre_validation_path, scripted_results)
pre_return = float(sum(result.total_reward for result in scripted_results) / len(scripted_results))
pre_length = max(len(result.rewards) for result in scripted_results)
logger.log_validation_trajectory(
    ValidationTrajectoryRecord(
        run_id=profile.run.name,
        step=0,
        task_id='pre-train-scripted-grpo',
        trajectory_path=str(pre_validation_path),
        return_sum=pre_return,
        episode_length=pre_length,
        success=None,
        metrics={
            'generations': len(scripted_results),
            'mean_total_reward': pre_return,
        },
    )
)
logger.log_artifact(RunArtifact(profile.run.name, str(pre_validation_path), 'validation_trajectory', step=0))
print('pre-train validation:', pre_validation_path)
print('mean return:', pre_return)


In [ ]:

from tunix_craftext.tunix import (
    build_agentic_grpo_cluster,
    load_agentic_grpo_qwen_assets,
    load_tunix_topology,
    pinned_qwen_tensor_shape,
    role_to_meshes,
    validate_agentic_grpo_preflight,
)
from tunix_craftext.models.tunix_adapter import load_qwen_hf_tokenizer


topology = load_tunix_topology(topology_config_path)
spec = profile.workload
validate_agentic_grpo_preflight(
    topology,
    spec,
    pinned_qwen_tensor_shape(),
    rollout_backend='vllm-offload',
)
meshes = role_to_meshes(topology)
logger.log_metric(
    MetricRecord(
        run_id=profile.run.name,
        step=0,
        split='benchmark',
        phase='preflight_passed',
        metrics={
            'visible_jax_devices': len(jax.devices()),
            'mini_batch_size': spec.mini_batch_size,
            'rollout_micro_batch_size': spec.rollout_micro_batch_size,
        },
    )
)
print('topology:', topology)
for role, mesh in meshes.items():
    print(role, mesh.shape, mesh.axis_names)
print('workload:', spec)


## Load actor/reference assets

Эта ячейка аллоцирует веса. После неё GPU memory уже занята actor/reference и vLLM rollout server может подняться при создании `RLCluster`/`GRPOLearner`.


In [ ]:

assets = load_agentic_grpo_qwen_assets(snapshot_path, topology)
hf_tokenizer = load_qwen_hf_tokenizer(snapshot_path)
print('actor:', type(assets.actor))
print('reference:', type(assets.reference))
print('tunix tokenizer:', type(assets.tokenizer))
print('hf tokenizer:', type(hf_tokenizer))


In [ ]:

from tunix.rl.agentic.agentic_grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.agentic.agents.tool_agent import ToolAgent
from tunix.rl.agentic.parser.chat_template_parser.parser import QwenChatTemplateParser

from tunix_craftext.env.agentic_craftext import (
    CrafTextAgenticEnvironment,
    CrafTextStepTool,
    agentic_environment_kwargs,
)

env_kwargs = agentic_environment_kwargs(environment_config_path)
cluster = build_agentic_grpo_cluster(topology, spec, assets, sync_vllm_generation)
learner = GRPOLearner(
    rl_cluster=cluster,
    algo_config=GRPOConfig(
        num_generations=profile.workload.num_generations,
        max_response_length=profile.workload.max_new_tokens,
        max_concurrency=profile.workload.max_concurrency,
        system_prompt='Use craftext_step for every environment action.',
    ),
    chat_parser=QwenChatTemplateParser(hf_tokenizer, enable_thinking=False),
    agent_class=ToolAgent,
    agent_kwargs={
        'system_prompt': 'Use craftext_step for every environment action.',
        'tool_parser_name': 'qwen',
        'tool_map': {'craftext_step': CrafTextStepTool},
    },
    env_class=CrafTextAgenticEnvironment,
    env_kwargs=env_kwargs,
)
logger.log_metric(
    MetricRecord(
        run_id=profile.run.name,
        step=0,
        split='benchmark',
        phase='learner_built',
        metrics={
            'rollout_engine_vllm': int(cluster.cluster_config.rollout_engine == 'vllm'),
            'max_response_length': profile.workload.max_new_tokens,
        },
        checkpoint_path=str(checkpoint_root_path),
    )
)
print('cluster:', type(cluster))
print('learner:', type(learner))
print('rollout engine:', cluster.cluster_config.rollout_engine)
print('checkpoint root:', cluster.cluster_config.training_config.checkpoint_root_directory)


In [ ]:
preview_batch = next(
    task_sampler.batches(
        seed=config.run.seed,
        batch_size=profile.workload.mini_batch_size,
        count=1,
    )
)
print('training batch preview:')
print(json.dumps({
    'goal': preview_batch['goal'],
    'seed': preview_batch['seed'].tolist(),
    'horizon': preview_batch['horizon'].tolist(),
    'instruction_index': preview_batch['instruction_index'].tolist(),
}, indent=2, ensure_ascii=False))

train_iterator = task_sampler.batches(
    seed=config.run.seed,
    batch_size=profile.workload.mini_batch_size,
    count=profile.workload.max_steps,
)
eval_iterator = task_sampler.batches(
    seed=config.run.seed + 10_000,
    batch_size=profile.workload.mini_batch_size,
    count=1,
)


## Real training update

Эта ячейка запускает настоящий Tunix `GRPOLearner.train(...)`: rollout через sync vLLM server-mode, group rewards, GRPO update actor weights, checkpoint root из workload. Если cell падает, ошибка логируется в `metrics.jsonl` и пробрасывается наружу.


In [ ]:

import time

train_started_at = time.perf_counter()
try:
    learner.train(train_iterator, eval_dataset=eval_iterator, skip_jit=False)
except Exception as error:
    elapsed_s = time.perf_counter() - train_started_at
    logger.log_metric(
        MetricRecord(
            run_id=profile.run.name,
            step=profile.workload.max_steps,
            split='train',
            phase='grpo_train_failed',
            metrics={'elapsed_s': elapsed_s, 'error_type_recorded': 1},
            checkpoint_path=str(checkpoint_root_path),
        )
    )
    raise
else:
    elapsed_s = time.perf_counter() - train_started_at
    logger.log_metric(
        MetricRecord(
            run_id=profile.run.name,
            step=profile.workload.max_steps,
            split='train',
            phase='grpo_train_finished',
            metrics={
                'elapsed_s': elapsed_s,
                'max_steps': profile.workload.max_steps,
                'num_generations': profile.workload.num_generations,
            },
            checkpoint_path=str(checkpoint_root_path),
        )
    )
    logger.log_artifact(
        checkpoint_artifact(
            profile.run.name,
            checkpoint_root_path,
            step=profile.workload.max_steps,
            role='tunix-rlcluster',
        )
    )
finally:
    cluster.close()

print('training finished in seconds:', elapsed_s)
print('checkpoint root:', checkpoint_root_path)


In [ ]:

post_validation_path = run_dir / 'validation' / 'post-train-scripted-grpo.json'
post_results = await collect_scripted_grpo_group(
    config_path=environment_config_path,
    goal=profile.run.goal,
    seed=profile.run.seed + 1,
    group_id=1,
    action_sequences=(('NOOP',) * 2, ('LEFT',) * 2),
    horizon=2,
)
save_scripted_grpo_smoke(post_validation_path, post_results)
post_return = float(sum(result.total_reward for result in post_results) / len(post_results))
post_length = max(len(result.rewards) for result in post_results)
logger.log_validation_trajectory(
    ValidationTrajectoryRecord(
        run_id=profile.run.name,
        step=profile.workload.max_steps,
        task_id='post-train-scripted-grpo',
        trajectory_path=str(post_validation_path),
        return_sum=post_return,
        episode_length=post_length,
        success=None,
        metrics={
            'generations': len(post_results),
            'mean_total_reward': post_return,
        },
    )
)
logger.log_artifact(
    RunArtifact(profile.run.name, str(post_validation_path), 'validation_trajectory', step=profile.workload.max_steps)
)
print('post-train validation:', post_validation_path)
print('mean return:', post_return)


In [ ]:
print('metrics path:', logger.metrics_path)
print('validation trajectories path:', logger.validation_trajectories_path)
print('artifacts path:', logger.artifacts_path)
print('\nmetrics tail:')
for row in read_jsonl(logger.metrics_path)[-10:]:
    print(json.dumps(row, ensure_ascii=False))
print('\nvalidation trajectory records:')
for row in read_jsonl(logger.validation_trajectories_path)[-10:]:
    print(json.dumps(row, ensure_ascii=False))
print('\nartifact records:')
for row in read_jsonl(logger.artifacts_path)[-10:]:
    print(json.dumps(row, ensure_ascii=False))
